In [2]:
import pandas as pd

customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
category_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

print("All files loaded successfully.")

All files loaded successfully.


In [3]:
print("orders shape:", orders.shape)
print("customers shape:", customers.shape)
print()
orders.info()

orders shape: (99441, 8)
customers shape: (99441, 5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [4]:
orders = pd.read_csv(
    "../data/raw/olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [5]:
print("order_status value counts:")
print(orders["order_status"].value_counts())
print()

print("unique order_id:", orders["order_id"].nunique())
print("total rows in orders:", len(orders))
print()

print("unique customer_id:", customers["customer_id"].nunique())
print("unique customer_unique_id:", customers["customer_unique_id"].nunique())

order_status value counts:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

unique order_id: 99441
total rows in orders: 99441

unique customer_id: 99441
unique customer_unique_id: 96096


### Key finding: customer_id vs customer_unique_id

`customer_id` is unique per row (99,441 unique = 99,441 rows), but `customer_unique_id` has
only 96,096 unique values. This means `customer_id` is generated fresh per order, while
`customer_unique_id` identifies the actual person. 

**Rule for this project: always use `customer_unique_id` for any repeat-customer or
repurchase-rate logic. Never use `customer_id` for that purpose.**

In [6]:
n_before = len(orders)

orders_delivered = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].notna()) &
    (orders["order_estimated_delivery_date"].notna())
].copy()

n_after = len(orders_delivered)

print(f"Orders before filtering: {n_before}")
print(f"Orders after filtering to delivered + valid dates: {n_after}")
print(f"Orders dropped: {n_before - n_after}")

Orders before filtering: 99441
Orders after filtering to delivered + valid dates: 96470
Orders dropped: 2971


In [7]:
orders_delivered["delivery_delay_days"] = (
    orders_delivered["order_delivered_customer_date"] - orders_delivered["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

orders_delivered["is_late"] = orders_delivered["delivery_delay_days"] > 0

print(orders_delivered["delivery_delay_days"].describe())
print()
print("% of orders delivered late:", round(orders_delivered["is_late"].mean() * 100, 1))

count    96470.000000
mean       -11.178126
std         10.184354
min       -146.016123
25%        -16.244065
50%        -11.948102
75%         -6.389815
max        188.975081
Name: delivery_delay_days, dtype: float64

% of orders delivered late: 8.1


In [8]:
reviews_sorted = order_reviews.sort_values("review_answer_timestamp")
reviews_dedup = reviews_sorted.drop_duplicates(subset="order_id", keep="last")

print("Reviews before dedup:", len(order_reviews))
print("Reviews after dedup:", len(reviews_dedup))
print("Unique order_ids after dedup:", reviews_dedup["order_id"].nunique())
assert reviews_dedup["order_id"].is_unique, "Dedup failed — still duplicates present!"
print("Confirmed: one review per order_id.")

Reviews before dedup: 99224
Reviews after dedup: 98673
Unique order_ids after dedup: 98673
Confirmed: one review per order_id.


In [9]:
items_agg = order_items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    n_distinct_sellers=("seller_id", "nunique")
).reset_index()

payments_agg = order_payments.groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    payment_methods_used=("payment_type", "nunique"),
    max_installments=("payment_installments", "max")
).reset_index()

print("items_agg shape:", items_agg.shape)
print("payments_agg shape:", payments_agg.shape)
assert items_agg["order_id"].is_unique
assert payments_agg["order_id"].is_unique
print("Confirmed: one row per order_id in both aggregated tables.")

items_agg shape: (98666, 5)
payments_agg shape: (99440, 4)
Confirmed: one row per order_id in both aggregated tables.


In [10]:
n_start = len(orders_delivered)
print("Starting rows:", n_start)

df = orders_delivered.merge(
    customers[["customer_id", "customer_unique_id", "customer_state"]],
    on="customer_id", how="left"
)
print("After customers join:", len(df))

df = df.merge(
    reviews_dedup[["order_id", "review_score", "review_answer_timestamp"]],
    on="order_id", how="left"
)
print("After reviews join:", len(df))

df = df.merge(items_agg, on="order_id", how="left")
print("After items join:", len(df))

df = df.merge(payments_agg, on="order_id", how="left")
print("After payments join:", len(df))

assert len(df) == n_start, "Row count changed unexpectedly during joins!"
print("\nConfirmed: row count unchanged through all joins.")
print("Final shape:", df.shape)

Starting rows: 96470
After customers join: 96470
After reviews join: 96470
After items join: 96470
After payments join: 96470

Confirmed: row count unchanged through all joins.
Final shape: (96470, 21)


In [11]:
print("Orders with no review:", df["review_score"].isna().sum())
print()

df["is_negative_review"] = df["review_score"] <= 2

has_review = df["review_score"].notna()
baseline = df.loc[has_review].groupby("is_late")["is_negative_review"].mean()

print("Negative review rate (score <= 2) by on-time vs late (only orders with a review):")
print(baseline)
print()
print("n on-time with review:", (has_review & ~df["is_late"]).sum())
print("n late with review:", (has_review & df["is_late"]).sum())

Orders with no review: 646

Negative review rate (score <= 2) by on-time vs late (only orders with a review):
is_late
False    0.092216
True     0.540660
Name: is_negative_review, dtype: float64

n on-time with review: 88163
n late with review: 7661


In [12]:
df.to_csv("../data/olist_orders_clean.csv", index=False)
print("Saved. Shape:", df.shape)

Saved. Shape: (96470, 22)


## Summary

This notebook builds the clean, order-level analysis table for the late-delivery experiment project.

**Key steps:**
- Loaded all 9 Olist tables and validated their structure (row counts, dtypes, nulls)
- Confirmed `customer_id` is generated per-order, while `customer_unique_id` identifies the actual customer (99,441 vs 96,096 unique values) — critical for any repeat-customer logic
- Filtered to delivered orders with valid delivery dates (96,470 of 99,441 orders retained)
- Engineered `delivery_delay_days` and `is_late`
- Deduplicated reviews (99,224 → 98,673), keeping the most recent response per order
- Aggregated item- and payment-level tables to order-level before joining
- Joined all tables into one clean table, validating row counts at every step (no fan-out)
- Computed the baseline result the project is built on: **9.2% negative review rate for on-time
  orders vs. 54.1% for late orders**
- Saved the result to `data/olist_orders_clean.csv` for use in later notebooks

**Next notebook:** power analysis and sample size calculation for the proposed experiment.